In [ ]:
import os
import subprocess

fasta_parts = [
    "data/training_data/animal_training_part1.fasta",
    "data/training_data/animal_training_part2.fasta",
    "data/training_data/animal_training_part3.fasta"
]
output_path = "data/aling_db/animal_training.fasta"

os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as outfile:
    for file in fasta_parts:
        with open(file, "r") as infile:
            outfile.write(infile.read().rstrip() + "\n")



In [ ]:
output_embbedins = output_path.replace(".fasta", "")+'_feats'

extract_esm2_embeddings_from_fasta(
    fasta_path=output_path,
    output_prefix=output_embbedins,
    batch_size=5
)

merge_and_cleanup_embeddings(output_prefix=output_embbedins)

In [4]:
import pandas as pd
import os

csv_parts = [
    "data/training_data/animal_training_part1.csv",
    "data/training_data/animal_training_part2.csv",
    "data/training_data/animal_training_part3.csv"
]

# Read and concatenate
dfs = [pd.read_csv(file) for file in csv_parts]
df_final = pd.concat(dfs, ignore_index=True)

# Save merged file
output_path_interactions = "data/training_data/animal_training_interactions.csv"
df_final.to_csv(output_path, index=False)

In [ ]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import gc

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import itertools
import random
import os



gc.collect()

df_feats = pd.read_csv(
    output_embbedins,
    header=None,
    dtype={0: str}  
)

df_feats.iloc[:, 1:] = df_feats.iloc[:, 1:].astype(np.float32)
df_feats = df_feats.rename(columns={0: "protein1"})
df_combined = pd.read_csv(output_path_interactions)

df_combined = df_combined[['protein1','protein2','Label']]
df_merged = df_combined.merge(df_feats, on="protein1", how="left")
df_feats = df_feats.rename(columns={'protein1': "protein2"})
df_merged = df_merged.merge(df_feats, on="protein2", how="left")
del df_feats
del df_combined

feature_cols = df_merged.columns[3:]
X = df_merged[feature_cols]
y = df_merged["Label"]
del df_merged
gc.collect()

X = X.astype(np.float32, copy=False)
scaler = StandardScaler(copy=False)
X_train_scaled = scaler.fit_transform(X)
X_train_scaled = X_train_scaled.astype(np.float32, copy=False)
joblib.dump(scaler, 'scaler_metazoa_900sc.pkl')
del X
gc.collect()


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout, batch_norm):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_one_fold(X_train, y_train, X_val, y_val, params, device):

    model = MLP(
        input_dim=X_train.shape[1],
        hidden_layers=params["hidden_layers"],
        dropout=params["dropout"],
        batch_norm=params["batch_norm"]
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=params["batch_size"],
        shuffle=True,
        pin_memory=True
    )

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(params["epochs"]):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # ---------- validation ----------
        model.eval()
        with torch.no_grad():
            logits = model(X_val.to(device))
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)

        f1 = f1_score(y_val.cpu().numpy(), preds)

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= params["patience"]:
            break

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return best_f1

RESULTS_FILE = "random_search_results.csv"

def save_result(row):
    df_row = pd.DataFrame([row])

    if os.path.exists(RESULTS_FILE):
        df_row.to_csv(RESULTS_FILE, mode="a", header=False, index=False)
    else:
        df_row.to_csv(RESULTS_FILE, index=False)




param_grid = {
    "hidden_layers": [
        [512, 256, 128],
        [1024, 512, 128],
        [1024, 512, 256, 128],
        [1024, 256, 128, 16],
        [512, 128]
    ],
    "dropout": [0.15, 0.2, 0.25],
    "batch_norm": [True, False],
    "lr": [1e-4, 3e-4, 1e-3, 3e-5],
    "weight_decay": [1e-4, 1e-5, 1e-6, 1e-7], 
    "batch_size": [4096, 8192],
    "epochs": [50, 100],
    "patience": [8]
}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = torch.tensor(X_train_scaled, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

del X_train_scaled
gc.collect()

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

all_combos = list(itertools.product(*param_grid.values()))
random.shuffle(all_combos)

N_TRIALS = 200   # ajuste aqui

keys = list(param_grid.keys())

for trial_id, combo in enumerate(all_combos[:N_TRIALS], 1):

    params = dict(zip(keys, combo))
    fold_scores = []

    for train_idx, val_idx in skf.split(X, y.cpu().numpy()):
        score = train_one_fold(
            X[train_idx], y[train_idx],
            X[val_idx], y[val_idx],
            params,
            device
        )
        fold_scores.append(score)

    result = {
        **params,
        "mean_f1": float(np.mean(fold_scores)),
        "std_f1": float(np.std(fold_scores))
    }

    save_result(result)

    print(f"[{trial_id}/{N_TRIALS}] F1 = {result['mean_f1']:.4f}")